# Introduction to Flask | Flask 简介

In this demo we will take show how to serve models with Flask as part of Model deployment.

在本演示中，我们将展示如何使用 Flask 作为模型部署的一部分来提供模型服务。

In [ ]:
# Install Flask | 安装 Flask
!pip install Flask

In [ ]:
# RESTART THE NOTEBOOK | 重启笔记本

In [ ]:
# Show the version | 显示版本
!python -m flask --version

In [ ]:
# Setup Structure for Flask App | 设置 Flask 应用的结构
!tree flask_app/

# Create a Flask App | 创建 Flask 应用

We will follow along in the cells alongside our `app.py` file.

我们将在单元格中跟随我们的 `app.py` 文件一起学习。

## Loading a model in Flask | 在 Flask 中加载模型

```py
# Initialize Flask app | 初始化 Flask 应用
app = Flask(__name__)

# Load any other environment variables or variables here | 在此处加载任何其他环境变量或变量
MY_SECRET = os.getenv('SECRET')

# Load the MobileNetV3 Large pre-trained model | 加载 MobileNetV3 Large 预训练模型
model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
model.eval()  # Set model to evaluation mode | 将模型设置为评估模式

```

## Create an endpoint | 创建端点

```py
# Map URL to an endpoint and method | 将 URL 映射到端点和方法
@app.route('/predict', methods=['POST'])
def predict():
    # Retrieve the request | 获取请求
    data = request.json
    
    # Transform input if needed | 如果需要，转换输入
    input_tensor = torch.tensor(data['input'])

    # Get the prediction | 获取预测
    output = model(input_tensor)
    
    # Return a response | 返回响应
    return jsonify({'output': output.tolist()})

```

## request

The `request` module in this code is part of Flask and is used to access data from incoming HTTP requests. 

此代码中的 `request` 模块是 Flask 的一部分，用于访问传入 HTTP 请求的数据。

Here, `request.json` parses the JSON payload sent with the POST request and converts it into a Python dictionary for easy access and processing.

在这里，`request.json` 解析 POST 请求中发送的 JSON 负载，并将其转换为 Python 字典，以便轻松访问和处理。

```py
from flask import request

@app.route('/predict', methods=['POST'])
def predict():
    # Retrieve the request | 获取请求
    data = request.json
```

## jsonify

`jsonify` is a Flask utility that converts Python dictionaries, lists, or other serializable objects into JSON format, which is the standard data format for API responses. 

`jsonify` 是 Flask 实用程序，可将 Python 字典、列表或其他可序列化对象转换为 JSON 格式，这是 API 响应的标准数据格式。

It also sets the correct Content-Type header (application/json) in the HTTP response, ensuring clients recognize the data as JSON.

它还在 HTTP 响应中设置正确的 Content-Type 头（application/json），确保客户端将数据识别为 JSON。

```py
from flask import jsonify

# Return a response | 返回响应
return jsonify({'output': output.tolist()})

```

## Health Endpoint | 健康检查端点

If you can get a response, the server is healthy and running.

如果您能够获得响应，则说明服务器健康且正在运行。

```py
# Health endpoint | 健康检查端点
@app.route('/health', methods=['GET'])
def health():
    """
    Health check endpoint to confirm the app is running.
    健康检查端点以确认应用正在运行。
    """
    return jsonify({'status': 'healthy'}), 200
```

## Logging | 日志记录

Logging in the app to see what is happening and see if/where errors occur.

在应用程序中记录日志，以查看正在发生的事情，并查看是否/在哪里发生错误。

```py
except Exception as e:
        logger.error(f"Error during prediction: {str(e)}")
        response = {'error': str(e)}
        logger.info(f"Response for /predict: {response}")
        
        return jsonify(response), 500
```

## Error Handling | 错误处理

A best practice for a Production deployment.

生产部署的最佳实践。

```py
# Prediction endpoint | 预测端点
@app.route('/predict', methods=['POST'])
def predict():
    # Error handling | 错误处理
    try:
        # Extract Base64 string from request JSON | 从请求 JSON 中提取 Base64 字符串
        data = request.json
        if 'image' not in data:
            # Return and error if there is no image in the request | 如果请求中没有图像则返回错误
            return jsonify({'error': 'No image provided'}), 400
        
        # Decode the Base64 image string | 解码 Base64 图像字符串
        image_data = base64.b64decode(data['image'])
        image = Image.open(io.BytesIO(image_data)).convert('RGB')
        
        # Preprocess the image | 预处理图像
        transformed_img = preprocess(image).unsqueeze(0)
        
        # Perform inference | 执行推理
        with torch.no_grad():
            output = model(transformed_img)
            _, predicted = torch.max(output.data, 1)
            print(predicted)
        
        # Return our prediction | 返回我们的预测
        return jsonify({'prediction': predicted.item()})
    
    # Fail with our error in the response | 在响应中返回错误
    except Exception as e:
        return jsonify({'error': str(e)}), 500
```

## Run Flask server (Development mode) | 运行 Flask 服务器（开发模式）

```bash
> python app.py
```

## Send a request | 发送请求

Sending binary data over HTTP.

通过 HTTP 发送二进制数据。

In [ ]:
import base64

with open('dog-1.jpg', 'rb') as img_file:
    base64_string = base64.b64encode(img_file.read()).decode('utf-8')
    print(base64_string)

In [ ]:
# Test a request using python requests library | 使用 python requests 库测试请求
import requests

# JSON payload with the Base64 encoded image | 包含 Base64 编码图像的 JSON 负载
payload = {
    "image": base64_string
}

# Set the headers | 设置头部
headers = {
    "Content-Type": "application/json"
}

In [ ]:
# Send POST request | 发送 POST 请求
response = requests.post("http://127.0.0.1:5000/predict", 
                         json=payload, 
                         headers=headers)

# Print the response | 打印响应
print("Status Code:", response.status_code)
print("Response JSON:", response.json())

In [ ]:
# Check the health endpoint | 检查健康检查端点
response = requests.get("http://127.0.0.1:5000/health")

# Print the response | 打印响应
print("Status Code:", response.status_code)
print("Response JSON:", response.json())

## Test Error Handling | 测试错误处理

Be sure to also check the logs.

请确保同时检查日志。

In [ ]:
# Test error handling | 测试错误处理
response = requests.post("http://127.0.0.1:5000/predict",
                         headers=headers)

# Print the response | 打印响应
print("Status Code:", response.status_code)
print("Response JSON:", response.json())

In [ ]:
# Send POST request | 发送 POST 请求
response = requests.post("http://127.0.0.1:5000/predict", 
                         json={"video": base64_string}, 
                         headers=headers)

# Print the response | 打印响应
print("Status Code:", response.status_code)
print("Response JSON:", response.json())

## Running Flask Server with Gunicorn | 使用 Gunicorn 运行 Flask 服务器

This is considered a better way to serve a Flask app.

这被认为是提供 Flask 应用服务的更好方法。

## Using Gunicorn | 使用 Gunicorn

```bash
> gunicorn -w 2 -b 0.0.0.0:8080 app:app
```

In [ ]:
## Send Request | 发送请求
# Send POST request | 发送 POST 请求
response = requests.post("http://127.0.0.1:8080/predict", 
                         json=payload, 
                         headers=headers)

# Print the response | 打印响应
print("Status Code:", response.status_code)
print("Response JSON:", response.json())

In [ ]:
import json 

# Downloaded from Hugging Face | 从 Hugging Face 下载
# https://huggingface.co/datasets/huggingface/label-files/blob/main/imagenet-1k-id2label.json
with open("labels.json", "r") as f:
    imagenet_classes = json.load(f)

In [ ]:
# Get the actual class name from our labels | 从标签中获取实际的类名
imagenet_classes['207']